# Detection To FoodLens Pipeline

This notebook connects multi-food detection outputs to the existing FoodLens classifier. The detector proposes candidate food regions; each crop is classified with the ResNet50 FT-V2 Food-101 model and then passed through the decision-layer logic.

This is the bridge notebook before adding multi-food outputs to the app.

## 1. Imports And Setup

All imports are centralized in this cell. The notebook expects crop outputs from Notebook 7 and FoodLens model artifacts from Kaggle inputs.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
import json
from pathlib import Path
from typing import Any

import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import models, transforms

## 2. Configuration

Set the detection output path and FoodLens artifact path here. Kaggle model and notebook-output paths may need to be adjusted after attaching inputs.

In [2]:
@dataclass(frozen=True)
class Config:
    SEED: int = 42
    NUM_CLASSES: int = 101
    DETECTION_OUTPUT_DIR: Path = Path(
        "/kaggle/input/notebooks/tuannm3823/"
        "07-multi-food-detection-exploration/"
        "results/multi_food_detection"
    )
    LOCAL_DETECTION_OUTPUT_DIR: Path = Path(
        "/kaggle/working/results/multi_food_detection"
    )
    DATA_DIR: Path = Path("/kaggle/input/datasets/kmader/food41")
    MODEL_ARTIFACT_DIR: Path = Path(
        "/kaggle/input/models/tuannm3823/food101-resnet50-refinements/"
        "pytorch/default/1"
    )
    APP_ARTIFACT_DIR: Path = Path(
        "/kaggle/input/notebooks/tuannm3823/"
        "06-demo-inference/results/food_recognition_demo"
    )
    CHECKPOINT_NAME: str = "resnet50_ft_v2_best.pth"
    CLASS_NAMES_FILE: str = "class_names.json"
    CALIBRATION_JSON_FILE: str = "calibration.json"
    DEFAULT_TEMPERATURE: float = 0.958111
    OUTPUT_DIR: Path = Path(
        "/kaggle/working/results/multi_food_foodlens_pipeline"
    )
    DETECTIONS_FILE: str = "detections.csv"
    IMAGE_SIZE: int = 224
    TOP_K: int = 5
    AUTO_CONFIDENCE: float = 0.70
    SUGGEST_CONFIDENCE: float = 0.35


CFG = Config()
torch.manual_seed(CFG.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.SEED)
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG


Config(SEED=42, NUM_CLASSES=101, DETECTION_OUTPUT_DIR=PosixPath('/kaggle/input/notebooks/tuannm3823/07-multi-food-detection-exploration/results/multi_food_detection'), LOCAL_DETECTION_OUTPUT_DIR=PosixPath('/kaggle/working/results/multi_food_detection'), DATA_DIR=PosixPath('/kaggle/input/datasets/kmader/food41'), MODEL_ARTIFACT_DIR=PosixPath('/kaggle/input/models/tuannm3823/food101-resnet50-refinements/pytorch/default/1'), APP_ARTIFACT_DIR=PosixPath('/kaggle/input/notebooks/tuannm3823/06-demo-inference/results/food_recognition_demo'), CHECKPOINT_NAME='resnet50_ft_v2_best.pth', CLASS_NAMES_FILE='class_names.json', CALIBRATION_JSON_FILE='calibration.json', DEFAULT_TEMPERATURE=0.958111, OUTPUT_DIR=PosixPath('/kaggle/working/results/multi_food_foodlens_pipeline'), DETECTIONS_FILE='detections.csv', IMAGE_SIZE=224, TOP_K=5, AUTO_CONFIDENCE=0.7, SUGGEST_CONFIDENCE=0.35)

## 3. Resolve Inputs

Resolve Notebook 7 crop outputs and FoodLens model artifacts using the same linked-artifact pattern as Notebook 6. The implementation keeps path handling in one place so the prediction cells stay focused on model inference.


In [3]:
def detection_artifact_roots() -> list[Path]:
    """Return likely roots for Notebook 7 detection artifacts."""
    roots = [CFG.DETECTION_OUTPUT_DIR, CFG.LOCAL_DETECTION_OUTPUT_DIR]

    notebook_root = Path("/kaggle/input/notebooks")
    if notebook_root.exists():
        roots.extend(sorted(notebook_root.glob("**/results/multi_food_detection")))

    return list(dict.fromkeys(roots))


def resolve_file(filename: str, roots: list[Path]) -> Path | None:
    """Resolve a file from known roots or recursive Kaggle input search."""
    candidates: list[Path] = []
    for root in roots:
        candidates.append(root / filename)
        if root.exists():
            candidates.extend(sorted(root.rglob(filename)))

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(sorted(kaggle_input.rglob(filename)))

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def resolve_detection_dir() -> Path:
    """Resolve the directory that contains Notebook 7 detection outputs."""
    detections_path = resolve_file(CFG.DETECTIONS_FILE, detection_artifact_roots())
    if detections_path is None:
        raise FileNotFoundError(
            f"Could not find {CFG.DETECTIONS_FILE}. Attach Notebook 7 outputs "
            "or run Notebook 7 in the same Kaggle session first."
        )
    return detections_path.parent


def resolve_crop_path(path_value: str, detection_dir: Path) -> Path:
    """Resolve crop paths saved by Notebook 7 across Kaggle runtimes."""
    crop_path = Path(path_value)
    if crop_path.exists():
        return crop_path

    candidate = detection_dir / "crops" / crop_path.name
    if candidate.exists():
        return candidate

    matches = sorted(detection_dir.rglob(crop_path.name))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not resolve crop image: {path_value}")


DETECTION_DIR = resolve_detection_dir()
DETECTIONS_PATH = DETECTION_DIR / CFG.DETECTIONS_FILE
print(f"Detection directory: {DETECTION_DIR}")
print(f"Detections file: {DETECTIONS_PATH}")


Detection directory: /kaggle/input/notebooks/tuannm3823/07-multi-food-detection-exploration/results/multi_food_detection
Detections file: /kaggle/input/notebooks/tuannm3823/07-multi-food-detection-exploration/results/multi_food_detection/detections.csv


## 4. Load FoodLens Classifier

The classifier is the existing ResNet50 FT-V2 Food-101 model loaded from the same Kaggle model path used in Notebook 6. Detection adds localization; it does not replace the classifier in this notebook.

In [4]:
def resolve_image_dir(data_dir: Path) -> Path:
    """Resolve Food-101 image directory for class-name fallback."""
    candidate_dirs = [data_dir / "images", data_dir]
    for candidate_dir in candidate_dirs:
        if not candidate_dir.exists():
            continue
        class_dirs = [path for path in candidate_dir.iterdir() if path.is_dir()]
        has_images = any(
            image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}
            for class_dir in class_dirs
            for image_path in class_dir.iterdir()
        )
        if has_images:
            return candidate_dir
    raise FileNotFoundError(f"Food-101 images were not found under {data_dir}.")


def read_json(path: Path, default: Any) -> Any:
    """Read JSON from disk with a fallback value.

    Args:
        path: JSON file path.
        default: Value returned when the path does not exist.

    Returns:
        Parsed JSON object or fallback value.
    """
    if not path.exists():
        return default
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def make_classifier_head(in_features: int) -> nn.Sequential:
    """Create the project-standard Food-101 classifier head."""
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, CFG.NUM_CLASSES),
    )


def load_class_names() -> list[str]:
    """Load class names from app artifacts or Food-101 folders."""
    class_names_path = resolve_file(CFG.CLASS_NAMES_FILE, [CFG.APP_ARTIFACT_DIR])
    if class_names_path is not None:
        class_names = read_json(class_names_path, [])
        if len(class_names) == 101:
            print(f"Class names: {class_names_path}")
            return class_names

    image_dir = resolve_image_dir(CFG.DATA_DIR)
    class_names = sorted(path.name for path in image_dir.iterdir() if path.is_dir())
    print(f"Class names: derived from {image_dir}")
    return class_names


def load_temperature() -> float:
    """Load calibration temperature from app artifacts when available."""
    calibration_path = resolve_file(CFG.CALIBRATION_JSON_FILE, [CFG.APP_ARTIFACT_DIR])
    calibration = read_json(calibration_path, {}) if calibration_path is not None else {}
    temperature = calibration.get("temperature", CFG.DEFAULT_TEMPERATURE)
    print(f"Temperature: {temperature}")
    return float(temperature)


def load_classifier() -> tuple[nn.Module, list[str], float, torch.device]:
    """Load the FoodLens classifier and metadata.

    Returns:
        Model, class names, calibration temperature, and device.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    class_names = load_class_names()
    if len(class_names) != 101:
        raise ValueError("class_names must contain 101 Food-101 labels.")

    checkpoint_path = resolve_file(CFG.CHECKPOINT_NAME, [CFG.MODEL_ARTIFACT_DIR])
    if checkpoint_path is None:
        raise FileNotFoundError(
            f"Could not find {CFG.CHECKPOINT_NAME} under Kaggle inputs."
        )

    model = models.resnet50(weights=None)
    model.fc = make_classifier_head(model.fc.in_features)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()

    print(f"Checkpoint: {checkpoint_path}")
    return model, class_names, load_temperature(), device


model, class_names, temperature, device = load_classifier()
print(f"Device: {device}")


Class names: /kaggle/input/notebooks/tuannm3823/06-demo-inference/results/food_recognition_demo/class_names.json
Checkpoint: /kaggle/input/models/tuannm3823/food101-resnet50-refinements/pytorch/default/1/resnet50_ft_v2_best.pth
Temperature: 0.9581114053726196
Device: cpu


## 5. Classify Detection Crops

Each crop receives top-k Food-101 predictions. The decision band is intentionally simple here; later app integration can reuse the full decision-policy artifacts.

In [5]:
preprocess = transforms.Compose(
    [
        transforms.Resize((CFG.IMAGE_SIZE, CFG.IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


def classify_crop(crop_path: Path) -> list[tuple[str, float]]:
    """Predict top-k Food-101 classes for one detected crop.

    Args:
        crop_path: Path to an exported crop image.

    Returns:
        Ordered `(class_name, confidence)` predictions.
    """
    image = Image.open(crop_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(image_tensor).cpu()
        probabilities = F.softmax(logits / temperature, dim=1)
        top_probabilities, top_indices = probabilities.topk(CFG.TOP_K, dim=1)
    return [
        (class_names[class_index], float(confidence))
        for class_index, confidence in zip(
            top_indices[0].tolist(),
            top_probabilities[0].tolist(),
        )
    ]


def decision_band(top_confidence: float) -> str:
    """Map top-1 confidence to a simple crop-level decision band."""
    if top_confidence >= CFG.AUTO_CONFIDENCE:
        return "auto_accept"
    if top_confidence >= CFG.SUGGEST_CONFIDENCE:
        return "suggest"
    return "confirm"

## 6. Export Multi-Food Predictions

The final table joins detector metadata with FoodLens crop predictions, then displays a compact review table for interpretation. Full detector metadata remains in the exported CSV for traceability.

In [6]:
detections_df = pd.read_csv(DETECTIONS_PATH)

prediction_rows: list[dict[str, object]] = []
for row in detections_df.itertuples(index=False):
    crop_path = resolve_crop_path(str(row.crop_path), DETECTION_DIR)
    predictions = classify_crop(crop_path)
    top_label, top_confidence = predictions[0]
    output = row._asdict()
    output["resolved_crop_path"] = str(crop_path)
    output.update(
        {
            "foodlens_top_label": top_label,
            "foodlens_top_confidence": top_confidence,
            "decision_band": decision_band(top_confidence),
            "top_k_predictions": json.dumps(predictions),
        }
    )
    prediction_rows.append(output)

multi_food_df = pd.DataFrame(prediction_rows)
multi_food_df.to_csv(CFG.OUTPUT_DIR / "multi_food_predictions.csv", index=False)

review_columns = [
    "source_id",
    "proposal_role",
    "detector_label",
    "detector_confidence",
    "foodlens_top_label",
    "foodlens_top_confidence",
    "decision_band",
    "resolved_crop_path",
]
review_df = multi_food_df[review_columns].sort_values(
    ["source_id", "foodlens_top_confidence"],
    ascending=[True, False],
)
review_df.to_csv(CFG.OUTPUT_DIR / "multi_food_review.csv", index=False)

decision_summary_df = (
    multi_food_df.groupby("decision_band", dropna=False)
    .size()
    .reset_index(name="crop_count")
    .sort_values("crop_count", ascending=False)
)
decision_summary_df.to_csv(
    CFG.OUTPUT_DIR / "multi_food_decision_summary.csv",
    index=False,
)

source_summary_df = (
    multi_food_df.groupby("source_id")
    .agg(
        crop_count=("source_id", "size"),
        top_foodlens_confidence=("foodlens_top_confidence", "max"),
        mean_foodlens_confidence=("foodlens_top_confidence", "mean"),
    )
    .reset_index()
)
source_summary_df.to_csv(
    CFG.OUTPUT_DIR / "multi_food_source_summary.csv",
    index=False,
)

print(f"Detected crops classified: {len(multi_food_df)}")
display(decision_summary_df)
display(source_summary_df)
display(review_df.head(20))


Detected crops classified: 23


,decision_band,crop_count
2,suggest,10
1,confirm,7
0,auto_accept,6


,source_id,crop_count,top_foodlens_confidence,mean_foodlens_confidence
0,sample_01_simplot_table,4,0.767625,0.640602
1,sample_02_party_food,2,0.240538,0.206004
2,sample_03_food_market,8,0.919981,0.529500
3,sample_04_orchard_table,5,0.740003,0.372501
4,sample_05_prohibition_table,4,0.971991,0.618757


,source_id,proposal_role,detector_label,detector_confidence,foodlens_top_label,foodlens_top_confidence,decision_band,resolved_crop_path
2,sample_01_simplot_table,serving_container,bowl,0.444506,ramen,0.767625,auto_accept,/kaggle/input/notebooks/tuannm3823/07-multi-fo...
0,sample_01_simplot_table,serving_container,bowl,0.865487,greek_salad,0.685543,suggest,/kaggle/input/notebooks/tuannm3823/07-multi-fo...
3,sample_01_simplot_table,serving_container,bowl,0.304340,hummus,0.643013,suggest,/kaggle/input/notebooks/tuannm3823/07-multi-fo...
1,sample_01_simplot_table,serving_container,bowl,0.481994,steak,0.466228,suggest,/kaggle/input/notebooks/tuannm3823/07-multi-fo...
4,sample_02_party_food,direct_food,cake,0.576349,falafel,0.240538,confirm,/kaggle/input/notebooks/tuannm3823/07-multi-fo...
5,sample_02_party_food,serving_container,bowl,0.304760,garlic_bread,0.171470,confirm,/kaggle/input/notebooks/tuannm3823/07-multi-fo...
13,sample_03_food_market,serving_container,bowl,0.304573,lasagna,0.919981,auto_accept,/kaggle/input/notebooks/tuannm3823/07-multi-fo...
12,sample_03_food_market,serving_container,bowl,0.398252,french_fries,0.752331,auto_accept,/kaggle/input/notebooks/tuannm3823/07-multi-fo...
8,sample_03_food_market,serving_container,bowl,0.669443,pad_thai,0.594489,suggest,/kaggle/input/notebooks/tuannm3823/07-multi-fo...
9,sample_03_food_market,serving_container,bowl,0.653762,fish_and_chips,0.571686,suggest,/kaggle/input/notebooks/tuannm3823/07-multi-fo...


## 7. App Integration Notes

The app should consume `multi_food_predictions.csv` or a matching API response shape later. Each detected region needs a source id, bounding box, resolved crop path, top-k FoodLens predictions, confidence, and decision band. Use `multi_food_review.csv` for human-readable inspection and `multi_food_decision_summary.csv` for product-level monitoring.